# Whole-image classification: with vs without segmentation (ROI)

Segmentation -> ROI -> classification on the whole images, compared with classifying all tissue tiles (no segmentation). Patient-level 5-fold cross-validation throughout.

1. Train the epithelium segmenter (5 rounds, ~40 min)
2. Segment all 1,461 whole images: each image by the round model that never saw its fold
3. Tile at 512x512 (blur Rahman images, Macenko normalise); two tile sets: **all** (>= 60% tissue) and **roi** (>= 50% epithelium)
4. Train ConvNeXt-Tiny 4-class for both settings x 5 rounds (10 runs)
5. Score each whole image by averaging its tiles; compare the two settings

Code: [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) · `scripts/segmentation/`, `scripts/data/tile_whole_images.py`, `scripts/classification/summarize_roi.py`

### Before running
- **Add Input** → the dataset made from `kaggle_upload/oral-cancer-whole-images.zip`.
- **Settings** → Accelerator **GPU T4**, Internet **On**.
- This takes **several hours**. Use **Save Version → Save & Run All (Commit)** so it keeps running when the browser is closed. Each step skips work that is already done, so a re-run resumes.
- When it finishes, download `roi_results.zip` from the version's **Output** tab (small; model weights are left out).

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

In [ ]:
try:
    import segmentation_models_pytorch
except ImportError:
    !pip install -q segmentation-models-pytorch

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

matches = list(Path("/kaggle/input").rglob("splits/whole_image_folds.csv"))
assert matches, "whole_image_folds.csv not found under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent
# Tiles (~6 GB) and model weights go to /tmp; only small results are copied to /kaggle/working
OUTPUT_ROOT = Path("/tmp/work")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Masks in repo:", len(list(Path(CODE_DIR, "annotations/epithelium/masks").glob("*.png"))))
!df -h /tmp | tail -1

## 2. Train the epithelium segmenter (~40 min)

In [ ]:
if (OUTPUT_ROOT / "runs/segmenter/summary.json").exists():
    print("Segmenter already trained, skipping")
else:
    !cd {CODE_DIR} && python -m scripts.segmentation.train_segmenter

## 3. Segment all whole images

In [ ]:
if (OUTPUT_ROOT / "splits/whole_image_masks.csv").exists():
    print("Masks already exist, skipping")
else:
    !cd {CODE_DIR} && python -m scripts.segmentation.segment_whole_images

## 4. Tile: all-tissue and ROI tile sets

In [ ]:
if (OUTPUT_ROOT / "splits/roi/roi_round5.csv").exists():
    print("Tiles already exist, skipping")
else:
    !cd {CODE_DIR} && python -m scripts.data.tile_whole_images

## 5. Train ConvNeXt-Tiny: 2 settings x 5 rounds

In [ ]:
import time

for setting in ["all", "roi"]:
    for k in range(1, 6):
        run = f"roi_{setting}_round{k}"
        if (OUTPUT_ROOT / "runs" / run / "metrics.json").exists():
            print(f"{run}: done, skipping")
            continue
        start = time.time()
        !cd {CODE_DIR} && python -u -m scripts.classification.train_classifier --csv splits/roi/{setting}_round{k}.csv --run-name {run} --epochs 15 --patience 5 2>&1 | grep -E "^Run:|Epoch|TEST|Balanced accuracy|Macro-F1|NDB-UFES test"
        print(f"{run}: {(time.time() - start) / 60:.0f} min")

## 6. Results (image level)

In [ ]:
!cd {CODE_DIR} && python -m scripts.classification.summarize_roi

## 7. Save results

`roi_results.zip`: segmenter scores and examples, whole-image masks (half resolution), tile CSVs, and for each classifier run its metrics, predictions, curves and confusion matrix. Model weights are left out (they are large and stay reproducible from the code).

In [ ]:
import zipfile

WORK = Path("/kaggle/working")
archive = WORK / "roi_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUTPUT_ROOT.rglob("*")):
        rel = f.relative_to(OUTPUT_ROOT)
        keep = (
            f.is_file()
            and f.suffix != ".pt"
            and (rel.parts[0] in ("runs", "segmentation")
                 or rel.as_posix() in ("splits/whole_image_masks.csv", "splits/whole_image_tiles.csv"))
        )
        if keep:
            zf.write(f, rel.as_posix())
print(archive, round(archive.stat().st_size / 2**20, 1), "MB")